# Vesuvius Surface Detection - Aggressive Topology

Maximum topology preservation targeting β₁ (holes/gaps).

**Changes from baseline (0.545):**
- T_low: 0.10 (very low - capture faint signals)
- T_high: 0.40 (low seeds)
- z_radius: 5 (aggressive Z closing)
- xy_radius: 2 (moderate XY closing)
- Per-slice binary_fill_holes (fills interior holes)
- dust: 100 (keep more small components)

In [ ]:
from IPython.display import clear_output

var="/kaggle/input/vsdetection-packages-offline-installer-only/whls"
!pip install \
  "$var"/keras_nightly-*.whl \
  "$var"/tifffile-*.whl \
  "$var"/imagecodecs-*.whl \
  "$var"/medicai-*.whl \
  --no-index \
  --find-links "$var"

clear_output()
print('Setup complete.')

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "jax"

import keras
from keras import ops
from medicai.transforms import (
    Compose,
    NormalizeIntensity
)
from medicai.models import TransUNet
from medicai.utils.inference import SlidingWindowInference

import numpy as np
import pandas as pd
import zipfile
import tifffile
import scipy.ndimage as ndi
from skimage.morphology import remove_small_objects

print(f'Keras backend: {keras.config.backend()}, version: {keras.version()}')

In [ ]:
import glob

root_dir = "/kaggle/input/vesuvius-challenge-surface-detection"
test_dir = f"{root_dir}/test_images"
output_dir = "/kaggle/working/submission_masks"
zip_path = "/kaggle/working/submission.zip"
os.makedirs(output_dir, exist_ok=True)

num_classes = 3
input_shape = (160, 160, 160)

# Find model weights - try multiple paths
model_path = None
candidates = [
    "/kaggle/input/vsd-model/keras/transunet/3/transunet.seresnext50.160px.comboloss.weights.h5",
    "/kaggle/input/vsd-transunet-weights/transunet.seresnext50.160px.comboloss.weights.h5",
]
for path in candidates:
    if os.path.exists(path):
        model_path = path
        break

if model_path is None:
    found = glob.glob("/kaggle/input/**/*comboloss*.h5", recursive=True)
    if found:
        model_path = found[0]
    else:
        raise FileNotFoundError("Model weights not found")

print(f'Model weights: {model_path}')

In [ ]:
test_df = pd.read_csv(f"{root_dir}/test.csv")
print(f'Test samples: {len(test_df)}')
test_df.head()

In [ ]:
def val_transformation(image):
    data = {"image": image}
    pipeline = Compose([
        NormalizeIntensity(
            keys=["image"],
            nonzero=True,
            channel_wise=False
        ),
    ])
    result = pipeline(data)
    return result["image"]

In [ ]:
# LB 0.545 config: comboloss weights, no activation (raw logits)
model = TransUNet(
    input_shape=(*input_shape, 1),
    encoder_name='seresnext50',
    classifier_activation=None,
    num_classes=num_classes,
)
model.load_weights(model_path)
print(f'Model params: {model.count_params() / 1e6:.1f}M')

In [ ]:
swi = SlidingWindowInference(
    model,
    num_classes=num_classes,
    roi_size=input_shape,
    sw_batch_size=1,
    mode='gaussian',
    overlap=0.5,
)
print(f'Sliding window: overlap=0.5, mode=gaussian')

In [ ]:
def load_volume(path):
    vol = tifffile.imread(path)
    vol = vol.astype(np.float32)
    vol = vol[None, ..., None]
    return vol

In [ ]:
def predict_with_tta(inputs, swi):
    logits = []

    # Original
    logits.append(swi(inputs))

    # Flips (spatial only)
    for axis in [1, 2, 3]:
        img_f = np.flip(inputs, axis=axis)
        p = swi(img_f)
        p = np.flip(p, axis=axis)
        logits.append(p)

    # Axial rotations (H, W)
    for k in [1, 2, 3]:
        img_r = np.rot90(inputs, k=k, axes=(2, 3))
        p = swi(img_r)
        p = np.rot90(p, k=-k, axes=(2, 3))
        logits.append(p)

    mean_logits = np.mean(logits, axis=0)
    mean_prob = ops.softmax(mean_logits, axis=-1)
    return mean_prob.argmax(-1).astype(np.uint8).squeeze()

In [ ]:
def build_anisotropic_struct(z_radius, xy_radius):
    z, r = z_radius, xy_radius
    if z == 0 and r == 0:
        return None
    if z == 0 and r > 0:
        size = 2 * r + 1
        struct = np.zeros((1, size, size), dtype=bool)
        cy, cx = r, r
        for dy in range(-r, r + 1):
            for dx in range(-r, r + 1):
                if dy * dy + dx * dx <= r * r:
                    struct[0, cy + dy, cx + dx] = True
        return struct
    if z > 0 and r == 0:
        struct = np.zeros((2 * z + 1, 1, 1), dtype=bool)
        struct[:, 0, 0] = True
        return struct
    depth = 2 * z + 1
    size = 2 * r + 1
    struct = np.zeros((depth, size, size), dtype=bool)
    cz, cy, cx = z, r, r
    for dz in range(-z, z + 1):
        for dy in range(-r, r + 1):
            for dx in range(-r, r + 1):
                if dy * dy + dx * dx <= r * r:
                    struct[cz + dz, cy + dy, cx + dx] = True
    return struct


def topo_postprocess(probs, T_low=0.10, T_high=0.40, z_radius=5, xy_radius=2, 
                     dust_min_size=100, use_fill_holes=True):
    # Step 1: 3D Hysteresis
    strong = probs >= T_high
    weak = probs >= T_low

    if not strong.any():
        return np.zeros_like(probs, dtype=np.uint8)

    struct_hyst = ndi.generate_binary_structure(3, 3)
    mask = ndi.binary_propagation(strong, mask=weak, structure=struct_hyst)

    if not mask.any():
        return np.zeros_like(probs, dtype=np.uint8)

    # Step 2: 3D Anisotropic Closing
    if z_radius > 0 or xy_radius > 0:
        struct_close = build_anisotropic_struct(z_radius, xy_radius)
        if struct_close is not None:
            mask = ndi.binary_closing(mask, structure=struct_close)

    # Step 3: Per-slice hole filling (targets β₁)
    if use_fill_holes:
        for z in range(mask.shape[0]):
            mask[z] = ndi.binary_fill_holes(mask[z])

    # Step 4: Dust removal
    if dust_min_size > 0:
        mask = remove_small_objects(mask.astype(bool), min_size=dust_min_size)

    return mask.astype(np.uint8)

In [ ]:
# Aggressive topology params
def inference_pipeline(volume):
    probs = predict_with_tta(volume, swi)
    final = topo_postprocess(
        probs,
        T_low=0.10,        # Very low - capture faint signals
        T_high=0.40,       # Low seeds
        z_radius=5,        # Aggressive Z closing
        xy_radius=2,       # Moderate XY closing
        dust_min_size=100, # Keep more small components
        use_fill_holes=True,  # Fill interior holes per slice
    )
    return final

In [ ]:
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for image_id in test_df["id"]:
        tif_path = f"{test_dir}/{image_id}.tif"
        
        volume = load_volume(tif_path)
        volume = val_transformation(volume)
        output = inference_pipeline(volume)
        
        out_path = f"{output_dir}/{image_id}.tif"
        tifffile.imwrite(out_path, output.astype(np.uint8))
        z.write(out_path, arcname=f"{image_id}.tif")
        os.remove(out_path)

print(f'Submission ZIP: {zip_path}')

In [ ]:
print('Submission verification:')
if os.path.exists(zip_path):
    zip_size = os.path.getsize(zip_path)
    print(f'  Size: {zip_size / 1024:.1f} KB')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        files = zf.namelist()
        print(f'  Files: {len(files)}')
        for f in files:
            info = zf.getinfo(f)
            print(f'    {f}: {info.compress_size/1024:.1f} KB')
    expected = set(test_df['id'].astype(str).tolist())
    actual = set([f.replace('.tif', '') for f in files])
    if expected == actual:
        print('  All test IDs present.')
    else:
        print(f'  WARNING - Missing: {expected - actual}')